# 03 — bghtrend: sweep review & walk-forward

Research artifact Fazy 2. Konwencja repo: notebook **cytuje pakiet** (`algo_bot.*`),
nigdy odwrotnie — kod produkcyjny żyje w `algo_bot/`.

Sekcje (wypełniane per sesja):
1. **Sweep review** (Sesja 4, 2026-07-04) — heurystyki A-E na `results/experiments/index.csv`
2. Walk-forward analysis (Sesja 5 — TBD)
3. Monte Carlo bootstrap (Sesja 6 — TBD)
4. Parameter stability heatmap (Sesja 6 — TBD)
5. Stress regimes (Sesja 7 — TBD)
6. MVP go/no-go summary (Sesja 8 — TBD)

## 1. Sweep review (Sesja 4)

### PRIOR — zapisany PRZED zobaczeniem wyników (2026-07-04, przed pilotem)

1. Większość z 30 sampli per config ma Sharpe_post ≤ 0; top-1 w przedziale 0.8–1.8; top-1 > 2.5 = podejrzenie lucky sample.
2. Microstructure kill największy na b3/15m (~0.3–0.6 Sharpe), na 4h ≤ 0.15.
3. b3/15m setki-tysiące trade'ów; b4/4h ryzyko n_trades < 100.
4. Clustering częściowy — realny test na tuning params (slope_thr_*, pullback_atr_mult, rr_target); klastrowanie w rr_target/trail_atr_mult, szum w t3_*.
5. Cross-symbol: częściowe pokrycie top-3 BTC/ETH.
6. Po filtrach (sharpe_post>1.5, PF>1.5, n_trades>100, DD>-0.20) zostaje 3–10 kandydatów, głównie 1h; scenariusz 0 kandydatów realny.

Interpretacja POST-sweep musi być porównana z tym priorem (dyscyplina: bez tego
dominuje post-hoc rationalization).

In [8]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from algo_bot.engine.walkforward import WF_ELIGIBILITY_THRESHOLDS

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INDEX_CSV = ROOT / "results" / "experiments" / "index.csv"
REVIEW_JSON = ROOT / "results" / "experiments" / "sweep_review.json"

idx = pd.read_csv(INDEX_CSV)
print(f"{len(idx)} wierszy, {idx['space_file'].nunique()} configów, kolumny OK: "
      f"{[c for c in ['sharpe_post','calmar_post','profit_factor_post','n_trades_post'] if c in idx.columns]}")

# params JSON -> kolumny p:*
params_df = pd.json_normalize(idx["params"].map(json.loads)).add_prefix("p:")
df = pd.concat([idx.reset_index(drop=True), params_df], axis=1)
GROUP = ["space_file", "symbol", "timeframe"]
df.groupby(GROUP).size().rename("n_samples").to_frame()

150 wierszy, 3 configów, kolumny OK: ['sharpe_post', 'calmar_post', 'profit_factor_post', 'n_trades_post']


n_samples
space_file       symbol   timeframe           
bghtrend_b1.yaml BTC/USDT 1h                30
                 ETH/USDT 1h                30
bghtrend_b3.yaml BTC/USDT 15m               30
bghtrend_b4.yaml BTC/USDT 4h                30
                 ETH/USDT 4h                30

### Przegląd grup — dystrybucja sharpe_post (Heurystyka B w pigułce)

In [9]:
def group_summary(g: pd.DataFrame) -> pd.Series:
    s = g["sharpe_post"].sort_values(ascending=False).to_numpy()
    return pd.Series({
        "n": len(g),
        "top1": s[0],
        "top5_mean": s[:5].mean(),
        "median": np.median(s),
        "pct_positive": (s > 0).mean(),
        "top1_top5_gap": s[0] - s[1:5].mean() if len(s) > 4 else np.nan,
        "raw_post_spread_mean": (g["sharpe_raw"] - g["sharpe_post"]).mean(),
    })

summary = df.groupby(GROUP).apply(group_summary, include_groups=False).round(3)
summary

n   top1  top5_mean  median  pct_positive  top1_top5_gap  raw_post_spread_mean
space_file       symbol   timeframe                                                                                   
bghtrend_b1.yaml BTC/USDT 1h         30.0  0.658      0.583   0.283         0.767          0.094                 0.039
                 ETH/USDT 1h         30.0  0.182     -0.001  -0.402         0.067          0.229                 0.030
bghtrend_b3.yaml BTC/USDT 15m        30.0  0.007     -0.212  -1.422         0.033          0.274                -0.021
bghtrend_b4.yaml BTC/USDT 4h         30.0  0.775      0.775     NaN         0.467          0.000                 0.004
                 ETH/USDT 4h         30.0  0.675      0.666   0.542         0.633          0.012                 0.004

### Filtry selekcyjne do walk-forward (Decyzja 5 kickoffu)

Wszystkie musi spełnić `WF_ELIGIBILITY_THRESHOLDS` (ADR-013, import z `algo_bot.engine.walkforward`):
`sharpe_post > 1.0`, `profit_factor_post > 1.3`, `n_trades_post > 100`, `max_drawdown_pct_post > -0.20`
(pre-WF filter, świadomie luźniejszy niż pierwotne arbitralne 1.5 — patrz ADR-013; NIE mylić z MVP go-live).
Sort composite: `sharpe_post * n_trades / 1000`.

In [10]:
MASK = (
    (df["sharpe_post"] > WF_ELIGIBILITY_THRESHOLDS["sharpe"])
    & (df["profit_factor_post"] > WF_ELIGIBILITY_THRESHOLDS["profit_factor"])
    & (df["n_trades_post"] > WF_ELIGIBILITY_THRESHOLDS["n_trades"])
    & (df["max_drawdown_pct_post"] > WF_ELIGIBILITY_THRESHOLDS["max_drawdown_pct"])
)
df["composite"] = df["sharpe_post"] * df["n_trades_post"] / 1000.0
candidates = df[MASK].sort_values("composite", ascending=False)
print(f"Kandydaci po filtrach: {len(candidates)}")
cols = GROUP + ["sharpe_post", "sharpe_raw", "profit_factor_post",
                "max_drawdown_pct_post", "n_trades_post", "composite", "run_id"]
candidates[cols]

Kandydaci po filtrach: 0


,space_file,symbol,timeframe,sharpe_post,sharpe_raw,profit_factor_post,max_drawdown_pct_post,n_trades_post,composite,run_id


### Heurystyka A — clustering w parameter space (top-10 per grupa)

Dla każdej grupy: value_counts parametrów **zmiennych w sweep space** wśród top-10.
Dominacja jednej wartości → parametr niesie sygnał; rozkład ~uniform → szum.
Interpretować względem taxonomy core/tuning
(`docs/reference/modules/strategy-bghtrend-pullback.md`).

In [11]:
def varying_params(g: pd.DataFrame) -> list[str]:
    return [c for c in g.columns if c.startswith("p:") and g[c].nunique(dropna=False) > 1]

def top_n(g: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    return g.sort_values("sharpe_post", ascending=False).head(n)

clustering: dict[str, dict] = {}
for key, g in df.groupby(GROUP):
    t10 = top_n(g)
    counts = {p.removeprefix("p:"): t10[p].value_counts().to_dict() for p in varying_params(g)}
    clustering["|".join(map(str, key))] = counts

# podgląd pierwszej grupy
first = next(iter(clustering))
print(first)
for p, vc in clustering[first].items():
    print(f"  {p}: {vc}")

bghtrend_b1.yaml|BTC/USDT|1h
  cooldown_bars: {5: 5, 20: 3, 10: 2}
  deadzone: {5.0: 5, 4.0: 3, 3.0: 2}
  ema_fast: {16: 2, 14: 2, 13: 1, 17: 1, 15: 1, 19: 1, 20: 1, 18: 1}
  ema_mid: {89: 8, 55: 2}
  entry_max_atr_mult: {0.65: 5, 0.6: 3, 0.75: 1, 0.55: 1}
  long_l1: {20: 6, 15: 4}
  long_l2: {15: 6, 10: 4}
  pullback_atr_mult: {0.12: 5, 0.13: 2, 0.11: 2, 0.1: 1}
  pullback_lookback: {20: 5, 10: 3, 15: 2}
  rr_target: {1.5: 6, 2.0: 4}
  short_l1: {5: 6, 7: 4}
  short_l2: {15: 6, 20: 4}
  short_l3: {10: 7, 15: 3}
  sl_atr_mult: {0.5: 3, 0.45: 3, 0.6: 3, 0.4: 1}
  slope_lookback: {21: 6, 34: 4}
  slope_thr_mid: {9e-05: 4, 8e-05: 3, 6e-05: 3}
  slope_thr_slow: {4.5e-05: 3, 3e-05: 3, 5e-05: 2, 6e-05: 1, 3.5e-05: 1}
  stale_max_bars: {60: 4, 40: 4, 30: 2}
  t3_b: {0.7: 6, 0.8: 3, 0.6: 1}
  t3_len: {6: 6, 5: 3, 4: 1}
  trail_atr_mult: {2.0: 4, 1.75: 3, 2.25: 2, 1.5: 1}


### Heurystyki B/C/D — kształt dystrybucji, raw vs post, n_trades sanity

In [12]:
detail: dict[str, dict] = {}
for key, g in df.groupby(GROUP):
    t10 = top_n(g)
    spread = (g["sharpe_raw"] - g["sharpe_post"])
    detail["|".join(map(str, key))] = {
        "sharpe_post_sorted": g["sharpe_post"].sort_values(ascending=False).round(3).tolist(),
        "raw_post_spread": {"mean": round(spread.mean(), 3), "std": round(spread.std(), 3),
                             "min": round(spread.min(), 3), "max": round(spread.max(), 3)},
        "top10_n_trades": t10["n_trades_post"].astype(int).tolist(),
        "top10_dd_post": t10["max_drawdown_pct_post"].round(3).tolist(),
        "top10_pf_post": t10["profit_factor_post"].round(3).tolist(),
    }

pd.DataFrame({k: {"spread_mean": v["raw_post_spread"]["mean"],
                  "spread_std": v["raw_post_spread"]["std"],
                  "top10_min_trades": min(v["top10_n_trades"]),
                  "top10_max_trades": max(v["top10_n_trades"])}
              for k, v in detail.items()}).T

,spread_mean,spread_std,top10_min_trades,top10_max_trades
bghtrend_b1.yaml|BTC/USDT|1h,0.039,0.021,1.0,80.0
bghtrend_b1.yaml|ETH/USDT|1h,0.030,0.017,6.0,151.0
bghtrend_b3.yaml|BTC/USDT|15m,-0.021,0.690,86.0,3131.0
bghtrend_b4.yaml|BTC/USDT|4h,0.004,0.001,1.0,2.0
bghtrend_b4.yaml|ETH/USDT|4h,0.004,0.004,1.0,3.0


### Heurystyka E — cross-symbol consistency (top-3 BTC vs ETH per config)

In [13]:
cross: dict[str, dict] = {}
for space, gs in df.groupby("space_file"):
    per_symbol = {}
    for sym, g in gs.groupby("symbol"):
        t3 = top_n(g, 3)
        per_symbol[sym] = [
            {p.removeprefix("p:"): row[p] for p in varying_params(gs)} |
            {"sharpe_post": round(row["sharpe_post"], 3)}
            for _, row in t3.iterrows()
        ]
    cross[space] = per_symbol

print(json.dumps(cross, indent=1, default=str)[:2000])

{
 "bghtrend_b1.yaml": {
  "BTC/USDT": [
   {
    "cooldown_bars": 5,
    "deadzone": 4.0,
    "ema_fast": 16,
    "ema_mid": 89,
    "entry_max_atr_mult": 0.75,
    "long_l1": 20,
    "long_l2": 15,
    "pullback_atr_mult": 0.13,
    "pullback_lookback": 10,
    "rr_target": 1.5,
    "short_l1": 5,
    "short_l2": 15,
    "short_l3": 15,
    "sl_atr_mult": 0.5,
    "slope_lookback": 34,
    "slope_thr_mid": 9e-05,
    "slope_thr_slow": 4.5e-05,
    "stale_max_bars": 60,
    "t3_b": 0.8,
    "t3_len": 6,
    "trail_atr_mult": 1.5,
    "sharpe_post": 0.658
   },
   {
    "cooldown_bars": 20,
    "deadzone": 3.0,
    "ema_fast": 13,
    "ema_mid": 89,
    "entry_max_atr_mult": 0.65,
    "long_l1": 20,
    "long_l2": 15,
    "pullback_atr_mult": 0.11,
    "pullback_lookback": 20,
    "rr_target": 1.5,
    "short_l1": 5,
    "short_l2": 20,
    "short_l3": 10,
    "sl_atr_mult": 0.5,
    "slope_lookback": 34,
    "slope_thr_mid": 8e-05,
    "slope_thr_slow": 5e-05,
    "stale_max_bars": 60

### Zrzut review JSON (artefakt do interpretacji poza notebookiem)

In [14]:
review = {
    "generated_at": pd.Timestamp.utcnow().isoformat(),
    "n_rows": int(len(df)),
    "groups": {"|".join(map(str, k)): int(len(g)) for k, g in df.groupby(GROUP)},
    "summary": json.loads(summary.reset_index().to_json(orient="records")),
    "candidates": json.loads(candidates[cols + ["params"]].to_json(orient="records")),
    "clustering_top10": clustering,
    "detail": detail,
    "cross_symbol_top3": cross,
}
REVIEW_JSON.write_text(json.dumps(review, indent=1, default=str))
print(f"zapisano {REVIEW_JSON} ({REVIEW_JSON.stat().st_size/1024:.0f} KB)")

zapisano /home/janek/quant_projects/algo_bot/results/experiments/sweep_review.json (23 KB)


/tmp/ipykernel_159534/2254565331.py:2: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "generated_at": pd.Timestamp.utcnow().isoformat(),


### Wnioski Sesji 4 (wypełnione po manual review)

*TBD — heurystyki A-E vs PRIOR, selekcja kandydatów WF, uzasadnienie.
Kanoniczna wersja w captains-log.*

## 2. Walk-forward analysis (Sesja 5 — TBD)

## 3. Monte Carlo bootstrap (Sesja 6 — TBD)

## 4. Parameter stability heatmap (Sesja 6 — TBD)

## 5. Stress regimes (Sesja 7 — TBD)

## 6. MVP go/no-go summary (Sesja 8 — TBD)